# 02 · Depth-Warp → triplane (TPV), step by step

**`TRIPLANE_LIFTERS["depth_warp"]` → `lifting.triplane.DepthWarpTriplaneLifter`**  ·  Harley et al., Simple-BEV `liftnet.py` (depth-based warp)

> Predict a depth distribution for every pixel, build a depth-weighted frustum per camera, and let every voxel PULL its feature from the frustum at its (u, v, depth) - then collapse along Z, Y and X.

This notebook shows, cell by cell, how the **Depth-Warp** lifting technique produces a
**tri-perspective view** (three orthogonal feature planes `xy` = BEV, `xz`, `yz`) instead of a single
bird's-eye-view map, *without changing a line of the existing BEV lifter* - the triplane module wraps it
(composition) and only adds what is needed to keep the height axis.

| § | what you will see |
|---|---|
| 1 | the idea, the animated step-by-step GIF and a detailed Mermaid diagram |
| 2 | every animation step as a still image |
| 3 | the shared toy scene (4 cameras, vehicles, pedestrians) |
| 4 | building the triplane lifter from the registry |
| 5 | **inside the lifter**: every intermediate tensor, with exact checks |
| 6 | the triplane contract, verified |
| 7 | end-to-end training on 3D semantic occupancy |
| 8 | the negative control: does it really use the cameras? |
| 9 | how to use it in your own code, recap, cheat sheet, exercises |

## 1. The idea

![Depth-Warp triplane lifting animation](../../assets/triplane/depth_warp.gif)


Simple-BEV smears a pixel along its whole ray. **Depth-Warp** fixes that with a learned **depth distribution**:
a `DepthHead` predicts, for every pixel, a softmax over `D` depth bins and a context feature. Their outer product is a
**frustum volume** `(C', D, H, W)` per camera: the context is placed mostly at the depths the network believes in.

Every voxel is then projected to `(u, v, depth)` and samples that frustum **trilinearly** ("pull" / backward warping).
A voxel only receives a strong feature if the network put probability mass at *its* depth.

The triplane version reuses `DepthWarpLifter.frustum_volume` and `DepthWarpLifter.warp` unchanged and swaps the
BEV compressor for `VoxelToTriplane`. Because the depth distribution concentrates features at the right range, the
`xz` / `yz` planes see sharp objects instead of rays.

### Detailed diagram

Colours: <span style="color:#d48806">■ data</span> · <span style="color:#1f77b4">■ fixed geometry</span>
· <span style="color:#8e44ad">■ learned</span> · <span style="color:#2e8b57">■ reused from the existing BEV lifter</span>
· <span style="color:#c0392b">■ outputs</span>. (Mermaid renders in JupyterLab ≥ 4.1, VS Code and on GitHub;
the matplotlib version below renders everywhere.)

```mermaid
flowchart LR
  IMG["N camera images<br/>(B, N, 3, H, W)"]:::data --> ENC["ImageEncoder<br/>pyramid (B, N, C, H_l, W_l)"]:::learned
  CAM["Cameras<br/>K (B,N,3,3) · cam_to_ego (B,N,4,4)"]:::data
  subgraph CORE["DepthWarpLifter (existing, unchanged)"]
    direction TB
    DH["DepthHead<br/>softmax over D bins + context C'"]:::learned --> FR["frustum_volume<br/>context ⊗ depth → (B,N,C',D,H,W)"]:::reused
    VC["voxel centres"]:::geo --> PRJ["Cameras.project<br/>(u, v, depth) per voxel"]:::geo
    FR --> WARP["warp: trilinear grid_sample<br/>of the frustum at (u, v, d)<br/>voxels PULL from frustums"]:::geo
    PRJ --> WARP
    WARP --> AVG["masked_camera_mean"]:::geo
  end
  ENC --> DH
  CAM --> PRJ
  AVG --> VOX["voxel volume (B, C', X, Y, Z)"]:::reused
  subgraph V2T["VoxelToTriplane (new)"]
    direction TB
    CXY["fold Z into channels<br/>HeightCompressor → xy"]:::learned
    CXZ["fold Y into channels<br/>HeightCompressor → xz"]:::learned
    CYZ["fold X into channels<br/>HeightCompressor → yz"]:::learned
  end
  VOX --> CXY & CXZ & CYZ
  CXY & CXZ & CYZ --> TPV
  TPV["TPVPlanes<br/>xy (B,C,X,Y) · xz (B,C,X,Z) · yz (B,C,Y,Z)"]:::out
  TPV --> BEVH["TriplaneBEVHead<br/>f = xy + xz + yz → fold Z<br/>BEV (B, C, X, Y)"]:::learned
  TPV --> AGG["TPVAggregator<br/>voxel logits (B,K,X,Y,Z)<br/>+ point logits (B,K,P)"]:::learned
  classDef data fill:#fff7e6,stroke:#d48806,color:#222
  classDef geo fill:#eaf2fb,stroke:#1f77b4,color:#222
  classDef learned fill:#f1e6f8,stroke:#8e44ad,color:#222
  classDef reused fill:#eaf7ea,stroke:#2e8b57,color:#222
  classDef out fill:#fdecea,stroke:#c0392b,color:#222
```

### Shapes and equations


$$ (\,p_\theta(d \mid u, v),\ c(u, v)\,) = \mathrm{DepthHead}(F_n), \qquad
\Phi_n(c', d, v, u) = c(u, v)_{c'} \cdot p_\theta(d \mid u, v) $$

$$ V(p) = \operatorname{mean}_{n:\,m_n(p)} \ \mathrm{trilinear}\big(\Phi_n,\ u_n(p),\ v_n(p),\ d_n(p)\big) $$

| tensor | shape |
|---|---|
| depth probabilities | `(B·N, D, h, w)`, sums to 1 over `D` |
| frustum `Φ` | `(B, N, C', D, h, w)` |
| voxel volume | `(B, C', X, Y, Z)` |

In [1]:
import os, sys, time, warnings
warnings.filterwarnings("ignore")

# tpv_kit lives next to this notebook; also works when Jupyter starts at the repo root.
for _p in (".", os.path.join("notebooks", "triplane")):
    if os.path.isdir(os.path.join(_p, "tpv_kit")):
        sys.path.insert(0, os.path.abspath(_p)); break

import numpy as np
import torch, torch.nn.functional as F
import matplotlib.pyplot as plt

from lifting import BaseLifter
from lifting.encoders import SimpleConvEncoder
from lifting.lifters import LIFTERS
from lifting.triplane import TRIPLANE_LIFTERS, build_triplane_lifter, build_triplane_model, build_triplane_bev_model
from tpv_kit import (TriplaneScene, PlanePlotter, PipelineDiagram, StoryboardViewer,
                     TriplaneContractChecker, OccupancyExperiment, ExperimentResult)

torch.manual_seed(0)
torch.set_num_threads(min(4, torch.get_num_threads()))
plt.rcParams.update({"figure.dpi": 100, "image.interpolation": "nearest"})
plotter = PlanePlotter()
print("torch", torch.__version__, "| triplane lifters:", TRIPLANE_LIFTERS.names())

torch 2.14.1 | triplane lifters: ['alpamayo', 'bev_adapter', 'bevformer', 'depth_warp', 'lift_splat', 'mlp_view', 'simple_bev', 'tiim', 'tpvformer']


In [2]:
PipelineDiagram('Depth-Warp → triplane').extend([
    ('DepthHead', 'softmax over D bins\n+ context', 'learned'),
    ('frustum', "context ⊗ depth\n(B,N,C',D,h,w)", 'reused'),
    ('project voxels', '(u, v, depth)\nper camera', 'geometry'),
    ('warp (pull)', 'trilinear sample\n+ camera mean', 'reused'),
    ('VoxelToTriplane', 'fold Z / Y / X', 'learned'),
    ('TPVPlanes', 'xy · xz · yz', 'learned'),
]).show()

## 2. The animation, one still per step

The GIF above is rendered by `lifting.triplane.animation` (Pillow only, exact geometry on a toy scene:
4 cameras at the origin, a car, a pedestrian and a pole). `StoryboardViewer` re-renders the **final frame of every
step** so the whole sequence can be read at once. Regenerate all GIFs with `python scripts/make_triplane_gifs.py`.

In [3]:
viewer = StoryboardViewer("depth_warp")
for i, title in enumerate(viewer.step_titles(), 1):
    print(f"step {i}: {title}")
viewer.contact_sheet()

step 1: Inputs
step 2: Per-pixel depth distribution
step 3: Frustum volume = depth ⊗ context
step 4: Pull: trilinear sample at (u, v, depth)
step 5: Collapse Z, Y and X
step 6: Triplane output


## 3. The scene

Every triplane notebook uses the same `TriplaneScene`: a `32 m × 32 m × 4 m` region cut into `16 × 16 × 4`
cells, four `48 × 96` cameras whose yaw is **randomly rotated per scene** (so no network can memorise where
"front" is - it must read the calibration), and procedurally placed vehicles (red) and pedestrians (blue).
The ground truth is a 3D occupancy grid `(B, X, Y, Z)`; collapsing it along each axis gives the three
**label planes** each triplane is supervised against.

In [4]:
scene = TriplaneScene()
print(scene.describe())
batch = scene.sample
plotter.cameras(batch.images, title="the four camera images of test scene 0")
plotter.labels([("ground truth", scene.label_planes(batch.occupancy))], title="what the three planes should encode")

grid      : x (-16.0, 16.0) y (-16.0, 16.0) z (-1.0, 3.0) m, resolution (16, 16, 4)
voxel     : (2.0, 2.0, 1.0) m
cameras   : 4 x (48, 96) px
scenes    : 128 train / 32 test (batch 4)
TPV cells : xy 256 + xz 64 + yz 64 = 384  (voxels: 1024)


## 4. Build the triplane lifter

Two equivalent ways:

* `TRIPLANE_LIFTERS.build("depth_warp", grid=..., in_channels=..., out_channels=...)` - the raw registry;
* `build_triplane_lifter("depth_warp", grid, channels, image_size, num_cameras, encoder)` - also injects the
  context some lifters need (`feature_size`, `num_cameras`, `depth_bins`), exactly like `lifting.models.build_lifter`
  does for BEV lifters.

The image encoder turns `(B·N, 3, H, W)` images into a feature pyramid (strides 4 and 8 here); every lifter
consumes `(B, N, C, H_l, W_l)` per level.

In [5]:
C = 32
encoder = SimpleConvEncoder(out_channels=C, num_levels=2)
features = scene.encode(encoder, batch)
cameras = batch.cameras
print("feature pyramid:", [tuple(f.shape) for f in features])

lifter = build_triplane_lifter("depth_warp", scene.grid, C, scene.image_size, scene.num_cameras, encoder)
print(type(lifter).__name__, "| is a BaseLifter too:", isinstance(lifter, BaseLifter))
print(f"parameters: {sum(p.numel() for p in lifter.parameters()):,}")
for name, child in lifter.named_children():
    n = sum(p.numel() for p in child.parameters())
    print(f"  .{name:<12s} {type(child).__name__:<28s} {n:>9,} params")

feature pyramid: [(4, 4, 32, 12, 24), (4, 4, 32, 6, 12)]
DepthWarpTriplaneLifter | is a BaseLifter too: True
parameters: 384,248
  .bev_head     TriplaneBEVHead                 37,984 params
  .core         DepthWarpLifter                 11,128 params
  .to_planes    VoxelToTriplane                335,136 params


## 5. Inside the lifter, step by step

### 5.1 Depth head and frustum volume

`lifter.frustum_volume(features)` = `DepthWarpLifter.frustum_volume` (unchanged). The depth bins are injected by
`build_triplane_lifter` from the grid (`DepthBins` covering the region).

In [6]:
bins = lifter.depth_bins
print(f"depth bins: {bins.num_bins} from {bins.min_depth} m to {bins.max_depth} m (step {bins.step:.2f} m)")
frustum = lifter.frustum_volume(features)              # (B, N, C', D, h, w)
print("frustum volume:", tuple(frustum.shape))
probs, ctx = lifter.core.depth_head(features[lifter.level].flatten(0, 1))
print("depth probs:", tuple(probs.shape), "| sums to 1:", torch.allclose(probs.sum(1), torch.ones(1)))
print("frustum == context ⊗ depth:", torch.allclose(frustum.flatten(0, 1), ctx[:, :, None] * probs[:, None], atol=1e-6))

depth bins: 24 from 1.0 m to 25.0 m (step 1.00 m)
frustum volume: (4, 4, 32, 24, 12, 24)
depth probs: (16, 24, 12, 24) | sums to 1: True
frustum == context ⊗ depth: True


### 5.2 Warp: every voxel pulls from the frustum at `(u, v, depth)`

Voxels that project outside the image or outside the depth range get no feature. Untrained, the depth distribution is
almost uniform, so the result still looks like Simple-BEV's rays; training sharpens it (see §7).

In [7]:
# How many cameras see every voxel centre? (the geometry every voxel-based lifter relies on)
pts = scene.grid.voxel_centers().view(1, -1, 3).expand(batch.images.shape[0], -1, -1)
uv, depth, valid = cameras.project(pts)                      # (B, N, X*Y*Z, 2), (B, N, P), (B, N, P)
seen = valid.sum(1)[0].view(*scene.grid.resolution).float() # cameras that see each voxel, scene 0
print("uv", tuple(uv.shape), "| fraction of voxels seen by >= 1 camera:", round((seen > 0).float().mean().item(), 3))
plotter.scalar_planes([seen.mean(2), seen.mean(1), seen.mean(0)],
                      "number of cameras that see each voxel, averaged along the collapsed axis", cmap="magma")

uv (4, 4, 1024, 2) | fraction of voxels seen by >= 1 camera: 0.988


In [8]:
voxels = lifter.lift_voxels(features, cameras)
print("voxel volume:", tuple(voxels.shape))
in_range = (valid & bins.contains(depth)).any(1)[0].float().mean().item()
print(f"voxels with a valid (pixel, depth) in at least one camera: {in_range:.1%}")

voxel volume: (4, 32, 16, 16, 4)
voxels with a valid (pixel, depth) in at least one camera: 98.8%


### Collapse the volume into three planes: `VoxelToTriplane`

A BEV lifter ends with a `HeightCompressor`: fold `Z` into the channels (`C·Z`) and convolve over `(X, Y)`.
`VoxelToTriplane` does that **three times**, each time moving a different axis to the end:

| plane | permute `(B, C, X, Y, Z)` → | folded axis | conv over |
|---|---|---|---|
| `xy` | `(B, C, X, Y, Z)` | `Z` | `(X, Y)` |
| `xz` | `(B, C, X, Z, Y)` | `Y` | `(X, Z)` |
| `yz` | `(B, C, Y, Z, X)` | `X` | `(Y, Z)` |

**Why fold rather than average?** Folding keeps *where along the collapsed axis* a feature was: the `xz` plane
can still tell a car on the left from one on the right. The next cell proves it on a toy volume.

In [9]:
from lifting.triplane import VoxelToTriplane
from lifting.geometry import GridSpec

tiny = GridSpec(resolution=(2, 3, 2))
to_planes = VoxelToTriplane(in_channels=1, grid=tiny, out_channels=4).eval()
a = torch.zeros(1, 1, 2, 3, 2); a[..., 0, 0, 1] = 1.0      # feature at y = 0 (left)
b = torch.zeros(1, 1, 2, 3, 2); b[..., 0, 2, 1] = 1.0      # same feature at y = 2 (right)
with torch.no_grad():
    pa, pb = to_planes(a), to_planes(b)
print("mean over Y identical?      ", torch.equal(a.mean(3), b.mean(3)), "  <- averaging would lose left vs right")
print("folded xz planes identical? ", torch.allclose(pa.xz, pb.xz), " <- VoxelToTriplane keeps it")
print("compressor weights (C_in·folded axis -> C_out):",
      {k: tuple(m.weight.shape) for k, m in to_planes.named_modules() if isinstance(m, torch.nn.Conv2d)})

mean over Y identical?       True   <- averaging would lose left vs right
folded xz planes identical?  False  <- VoxelToTriplane keeps it
compressor weights (C_in·folded axis -> C_out): {'compress_xy.conv.0.0': (4, 2, 3, 3), 'compress_xy.conv.1': (4, 4, 1, 1), 'compress_xz.conv.0.0': (4, 3, 3, 3), 'compress_xz.conv.1': (4, 4, 1, 1), 'compress_yz.conv.0.0': (4, 2, 3, 3), 'compress_yz.conv.1': (4, 4, 1, 1)}


In [10]:
voxels = lifter.lift_voxels(features, cameras)
planes = lifter.to_planes(voxels)
print("voxels", tuple(voxels.shape), "->", {k: tuple(getattr(planes, k).shape) for k in ("xy", "xz", "yz")})
print(f"memory: voxels {voxels[0].numel():,} floats per scene vs planes "
      f"{sum(getattr(planes, k)[0].numel() for k in ('xy', 'xz', 'yz')):,}")

voxels (4, 32, 16, 16, 4) -> {'xy': (4, 32, 16, 16), 'xz': (4, 32, 16, 4), 'yz': (4, 32, 16, 4)}
memory: voxels 32,768 floats per scene vs planes 12,288


## 6. The triplane contract, verified

Every triplane lifter promises the same five properties (Liskov: it is still a `BaseLifter`, so it drops into
`BEVSegmentationModel`, the trainer and the verifier unchanged). `TriplaneContractChecker` runs them on real tensors.

In [11]:
assert TriplaneContractChecker(lifter, features, cameras).run()
with torch.no_grad():
    planes = lifter.lift_planes(features, cameras)
plotter.features(planes, title="untrained planes (joint PCA → RGB): structure comes only from geometry + random weights")

PASS  plane shapes            xy(4, 32, 16, 16) xz(4, 32, 16, 4) yz(4, 32, 16, 4)
PASS  BEV = bev_head(planes)  BEV (4, 32, 16, 16), max |diff| = 0.0e+00
PASS  f = xy + xz + yz        voxel (8,8,2) max |diff| = 0.0e+00
PASS  reuses BEV lifter       core = DepthWarpLifter = LIFTERS['depth_warp']
PASS  gradients reach images  1/2 pyramid levels receive gradient


## 7. Train end to end on 3D semantic occupancy

`build_triplane_model("depth_warp", ...)` = encoder → **this triplane lifter** → `TPVAggregator` (an MLP on
`f(x,y,z) = xy + xz + yz`), plus one auxiliary classifier per plane. `OccupancyTask(aux_weight=0.5)` adds a
cross-entropy on each plane against the collapsed labels of §3. 150 AdamW steps on 128 scenes, CPU only.

In [12]:
exp = OccupancyExperiment(scene, "depth_warp")
result = exp.run(steps=150)
print(ExperimentResult.header()); print(result.row())
OccupancyExperiment.plot_losses([result], title="Depth-Warp triplane: occupancy training loss")

lifter          params    time   before    after  corrupted   verdict
depth_warp        521k   26.7s    0.004    0.386      0.029   uses geometry


In [13]:
out = exp.predict()
pred = out["voxel_logits"].argmax(1)                       # (B, X, Y, Z) class ids
plotter.labels([("ground truth", scene.label_planes(batch.occupancy)),
                ("prediction", scene.label_planes(pred))],
               title="held-out scene: occupancy collapsed onto the three planes")
plotter.features(out["planes"], title="trained feature planes (joint PCA → RGB)")

### What did the depth head learn?

The depth head is trained **only through the occupancy loss** - there is no depth label. Two diagnostics on the test
scene: the mean **entropy** of the per-pixel distribution (uniform over 24 bins = ln 24 ≈ 3.18) and the correlation
between the **expected depth** `E[d] = Σ p(d)·d` and the rendered ground-truth depth.

In this short run the distribution stays close to uniform and `E[d]` barely tracks the true depth: occupancy
supervision alone hardly shapes depth here, so Depth-Warp's volume still behaves much like Simple-BEV's ray smearing
(and reaches a similar mIoU). This is why depth-based lifters are usually trained with an explicit depth loss - the
next section adds one and measures the difference.

In [14]:
from lifting.training.depth_supervision import nearest_surface_depth

trained = exp.model.lifter
with torch.no_grad():
    feats = scene.encode(exp.model.encoder, batch)
    probs, _ = trained.core.depth_head(feats[trained.level].flatten(0, 1))     # (B*N, D, h, w)
h, w = probs.shape[-2:]
gt = nearest_surface_depth(batch.depth.flatten(0, 1), (h, w))   # nearest surface per feature pixel
centers = trained.depth_bins.centers()
expected = (probs * centers[None, :, None, None]).sum(1)
mask = trained.depth_bins.contains(gt)
entropy = -(probs * probs.clamp_min(1e-9).log()).sum(1).mean().item()
corr = torch.corrcoef(torch.stack([expected[mask], gt[mask]]))[0, 1].item()
print(f"mean entropy {entropy:.3f} (uniform = {np.log(len(centers)):.3f}) | corr(E[d], true depth) = {corr:+.3f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.2), facecolor="white")
axes[0].scatter(gt[mask], expected[mask], s=3, alpha=0.3, color="#8e44ad")
axes[0].plot([1, 25], [1, 25], "k--", lw=1); axes[0].set_xlabel("true depth (m)"); axes[0].set_ylabel("E[d] (m)")
axes[0].set_title("expected vs true depth, all pixels of the test batch", fontsize=10)
k = int(torch.argmin(torch.where(mask, gt, torch.full_like(gt, 1e9)).flatten()))   # nearest object pixel
n, v, u = np.unravel_index(k, gt.shape)
axes[1].bar(centers.numpy(), probs[n, :, v, u].numpy(), width=trained.depth_bins.step * 0.9, color="#8e44ad")
axes[1].axvline(gt[n, v, u].item(), color="k", ls="--"); axes[1].set_xlabel("depth (m)")
axes[1].set_title(f"distribution at the nearest surface pixel (dashed = true depth)", fontsize=10)
plt.tight_layout(); plt.show()

mean entropy 3.105 (uniform = 3.178) | corr(E[d], true depth) = -0.343


### Training with an explicit depth loss

`Depth-Warp` above learned its depth distribution **only through the occupancy loss**. Here the same model, data,
seed and 150 steps are trained with `lifting.training.DepthSupervisedTask`, which wraps `OccupancyTask` and adds a
cross-entropy between every `DepthHead`'s bins and the rendered depth (BEVDepth, Li et al. AAAI 2023,
[arXiv:2206.10092](https://arxiv.org/abs/2206.10092)). It hooks the existing `DepthHead`, so the lifter is unchanged.

Diagnostics on the held-out batch: mean **entropy** of the per-pixel distribution (uniform over 24 bins =
ln 24 ≈ 3.18) and the correlation between **expected depth** `E[d] = Σ p(d)·d` and the true nearest-surface depth.

**Weight.** The depth term is weighted `0.05`. A sweep over `{0, 0.05, 0.1, 0.3, 1.0}` x `{150, 400}` steps on this
scene showed: every non-zero weight teaches depth (corr > 0.9), but weights ≥ 0.1 cost occupancy mIoU here because
depth then dominates the 150-step budget; at `0.05` mIoU stays within noise of the occupancy-only run.


In [15]:
from lifting.training import DepthSupervisedTask, OccupancyTask
from lifting.training.depth_supervision import DEPTH_PREDICTIONS

exp_depth = OccupancyExperiment(scene, "depth_warp", depth_weight=0.05)  # same seed / data / steps
result_depth = exp_depth.run(steps=150)
result_depth.name = "+depth loss"
print(ExperimentResult.header()); print(result.row()); print(result_depth.row())


def depth_diagnostics(experiment):
    """Entropy and corr(E[d], true depth) of the trained DepthHead on the held-out batch."""
    probe = DepthSupervisedTask(OccupancyTask(3))          # only used to capture the DepthHead output
    with torch.no_grad():
        (probs, bins), = probe.forward(experiment.model.eval(), batch)[DEPTH_PREDICTIONS]
    return DepthSupervisedTask.entropy_and_correlation(probs, batch.depth.flatten(0, 1), bins)


print()
for label, experiment in (("occupancy only", exp), ("+ depth loss", exp_depth)):
    entropy, corr = depth_diagnostics(experiment)
    print(f"{label:15s} mean entropy {entropy:.3f} (uniform = {np.log(24):.3f}) | corr(E[d], true depth) = {corr:+.3f}")


lifter          params    time   before    after  corrupted   verdict
depth_warp        521k   26.7s    0.004    0.386      0.029   uses geometry
+depth loss       521k   21.1s    0.004    0.387      0.029   uses geometry

occupancy only  mean entropy 3.105 (uniform = 3.178) | corr(E[d], true depth) = -0.343
+ depth loss    mean entropy 2.142 (uniform = 3.178) | corr(E[d], true depth) = +0.925


## 8. Negative control: does it really use the cameras?

Same images, same weights - but every camera's extrinsic is rotated by 90° about the vertical axis. A lifter that reads
pixels **through** the calibration now puts every object in the wrong place and its mIoU collapses; a geometry-free
lifter would not notice. (This is the triplane version of the repo's `lifting-verify` check.)

In [16]:
corrupted = exp.predict(corrupted=True)["voxel_logits"].argmax(1)
plotter.labels([("ground truth", scene.label_planes(batch.occupancy)),
                ("correct calib.", scene.label_planes(pred)),
                ("cams yawed 90°", scene.label_planes(corrupted))],
               title="same weights, wrong extrinsics")
print(f"mIoU  correct calibration: {result.miou_after:.3f}   |   cameras yawed 90°: {result.miou_corrupted:.3f}"
      f"   ->  {'uses geometry ✅' if result.uses_geometry else 'does NOT use geometry'}")

mIoU  correct calibration: 0.386   |   cameras yawed 90°: 0.029   ->  uses geometry ✅


## 9. Use it in your own code

```python
from lifting.triplane import TRIPLANE_LIFTERS, build_triplane_model, build_triplane_bev_model

# 1) just the lifter: features (B, N, C, H_l, W_l) per level + Cameras
lifter = TRIPLANE_LIFTERS.build("depth_warp", grid=grid, in_channels=64, out_channels=64)
planes = lifter.lift_planes(features, cameras)   # TPVPlanes(xy, xz, yz)
bev    = lifter(features, cameras)               # (B, 64, X, Y) - drop-in BEV lifter

# 2) 3D semantic occupancy + point queries
model = build_triplane_model("depth_warp", grid, num_classes=3, aux_heads=True)
out = model(images, cameras, points=lidar_xyz)   # planes, voxel_logits, point_logits, aux_logits

# 3) BEV segmentation read off the triplane (same API as lifting.build_bev_model)
bev_model = build_triplane_bev_model("depth_warp", grid, num_classes=3)
```

In [17]:
bev_model = build_triplane_bev_model("depth_warp", scene.grid, 3, scene.image_size, scene.num_cameras, channels=16)
with torch.no_grad():
    logits = bev_model(batch.images, batch.cameras)["logits"]
print("BEV segmentation logits read off the triplane:", tuple(logits.shape))

BEV segmentation logits read off the triplane: (4, 3, 16, 16)


## Recap


* Depth-Warp = Simple-BEV + a learned per-pixel depth distribution; voxels **pull** from a depth-weighted frustum.
* The frustum and the warp are the BEV lifter's, unchanged; only `VoxelToTriplane` is new.
* Without a depth label, occupancy supervision alone barely shapes the depth distribution in a short run; `DepthSupervisedTask` (a BEVDepth-style depth cross-entropy) teaches it, at no occupancy cost with a small weight.


**Cheat sheet**

| concept | file |
|---|---|
| depth head | `src/lifting/lifters/depth_head.py` |
| depth supervision | `src/lifting/training/depth_supervision.py` |
| frustum + warp (unchanged) | `src/lifting/lifters/depth_warp_lifter.py` |
| depth bins | `src/lifting/geometry/depth_bins.py` |
| triplane lifter | `src/lifting/triplane/depth_warp_triplane.py` |
| triplane contract | `src/lifting/triplane/base.py` |
| BEV read-off from planes | `src/lifting/triplane/triplane_bev_head.py` |
| occupancy model / factories | `src/lifting/triplane/triplane_occupancy_model.py`, `factory.py` |
| this animation | `src/lifting/triplane/animation/storyboards/depth_warp_storyboard.py` |

**Exercises**

1. Use a sharp softmax (`temperature=0.07`, Simple-BEV's choice). How do the planes look before training?
2. Train longer (e.g. 1000 steps) with and without `depth_weight`: does the sharper depth distribution start to pay off in occupancy mIoU?
3. Compare the number of parameters with `simple_bev` - where do they come from?

Next: see `00_triplane_overview.ipynb` for all techniques side by side.